# ProverbGap: Shortcut Audit Fix (Pilot v2)
This notebook implements the v2 prompts and the **Shortcut Resistance Score (SRS)** analysis.
It uses all available free-tier API keys via Kaggle Secrets (Groq x8, Cerebras x2, Gemini x4) with a smart rotation system to maximize throughput and avoid rate limits.

**Target**: Generate 50 items per language, then audit them. An SRS below 40% (close to 25% random chance) means the shortcut is eliminated and we are ready to scale to 1,000 items.


In [ ]:
# Cell 1: Setup and API Key Loading
import subprocess, sys, time, os, json, re, random, threading
import requests, zipfile, glob
from pathlib import Path
!pip install -q pandas requests
import pandas as pd

def _load_secrets():
    keys = {}
    try:
        from kaggle_secrets import UserSecretsClient
        s = UserSecretsClient()
        def get(name):
            try: 
                val = s.get_secret(name)
                return val if val and val.strip() else None
            except: return None
            
        keys['GROQ'] = [get(f'GROQ_API_KEY{"_"+str(i) if i>1 else ""}') for i in range(1,9)]
        keys['GROQ'] = [k for k in keys['GROQ'] if k]
        
        keys['CEREBRAS'] = [get(f'CEREBRAS_API_KEY{"_"+str(i) if i>1 else ""}') for i in range(1,3)]
        keys['CEREBRAS'] = [k for k in keys['CEREBRAS'] if k]
        
        keys['GEMINI'] = [get(f'GEMINI_API_KEY{"_"+str(i) if i>1 else ""}') for i in range(1,5)]
        keys['GEMINI'] = [k for k in keys['GEMINI'] if k]
        
        keys['NVIDIA'] = [get(f'NVIDIA_API_KEY{"_"+str(i) if i>1 else ""}') for i in range(1,5)]
        keys['NVIDIA'] = [k for k in keys['NVIDIA'] if k]
    except ImportError:
        print("Not running on Kaggle. Add keys to environment variables.")
        keys['GROQ'] = [os.environ.get('GROQ_API_KEY')] if os.environ.get('GROQ_API_KEY') else []
        keys['CEREBRAS'] = [os.environ.get('CEREBRAS_API_KEY')] if os.environ.get('CEREBRAS_API_KEY') else []
        keys['GEMINI'] = [os.environ.get('GEMINI_API_KEY')] if os.environ.get('GEMINI_API_KEY') else []
        keys['NVIDIA'] = [os.environ.get('NVIDIA_API_KEY')] if os.environ.get('NVIDIA_API_KEY') else []
        
    return keys

KEYS = _load_secrets()
print(f"Loaded Keys:")
for k, v in KEYS.items():
    print(f"  {k}: {len(v)} keys loaded")

OUT_DIR = Path('/kaggle/working')
OUT_DIR.mkdir(exist_ok=True)
print("Setup complete.")


In [ ]:
# Cell 2: Prompts v2
# These prompts were designed based on a 3-reviewer synthesis to eliminate stylistic and length-based shortcuts.

SYS_B_V2 = (
    'You are building a rigorous benchmark to test cultural understanding of proverbs. '
    'Given a proverb and its correct cultural meaning, generate 3 ALTERNATIVE cultural '
    'interpretations. The alternatives must:\n'
    '1. Be written in the SAME cultural register as the correct meaning — '
    'not more academic, not more casual, not more abstract\n'
    '2. Be SIMILAR in length to the correct meaning\n'
    '3. Express a DIFFERENT life lesson, social value, or communal principle '
    'than the correct meaning\n'
    '4. Sound equally plausible as genuine cultural wisdom — '
    'a reader who does not know this proverb should find all 4 options credible\n'
    '5. Be wrong because they convey a different cultural message, '
    'NOT because they sound absurd, obviously off-topic, or literal\n'
    'The 3 alternatives should be indistinguishable in format and tone from the '
    'correct meaning. Return ONLY a valid JSON list of 3 strings. No markdown.'
)

def make_prompt_b_v2(proverb, translation, cultural_meaning, lang):
    yoruba_note = ''
    if lang == 'Yoruba':
        yoruba_note = (
            '\nIMPORTANT: Each alternative must reflect a specific Yoruba social value '
            'or communal principle — not a generic life lesson. Ground each alternative '
            'in Yoruba community life, family structure, or social hierarchy.'
        )
    return (
        f'Proverb ({lang}): {proverb}\n'
        f'English translation: {translation}\n'
        f'Correct cultural meaning: {cultural_meaning}'
        f'{yoruba_note}\n\n'
        'Generate 3 alternative cultural interpretations where each:\n'
        '- Is written in the same style and cultural register as the correct meaning above\n'
        '- Is similar in length to the correct meaning above\n'
        '- Conveys a DIFFERENT life lesson or social value\n'
        '- Sounds equally legitimate as genuine cultural wisdom\n'
        '- Is wrong because it misattributes the cultural message — not because it sounds absurd\n\n'
        'Return ONLY: ["alternative 1", "alternative 2", "alternative 3"]'
    )

SYS_A_V2 = (
    'You are building a rigorous benchmark to test literal comprehension of proverbs. '
    'Given a proverb and its correct English translation, generate 3 ALTERNATIVE English '
    'translations. The alternatives must:\n'
    '1. Be written in the SAME natural translation register as the correct translation\n'
    '2. Be SIMILAR in length to the correct translation\n'
    '3. Introduce a SUBTLE meaning shift — different agent, direction, object, or condition '
    'compared to the correct translation\n'
    '4. Sound like genuine direct translations of a proverb\n'
    '5. Be grammatically natural English — not broken or obviously wrong\n'
    'Return ONLY a valid JSON list of 3 strings. No markdown.'
)

def make_prompt_a_v2(proverb, translation, lang):
    return (
        f'Proverb ({lang}): {proverb}\n'
        f'Correct English translation: {translation}\n\n'
        'Generate 3 alternative translations where each:\n'
        '- Is written in the same natural translation style as the correct translation above\n'
        '- Is similar in length to the correct translation above\n'
        '- Introduces a subtle meaning difference (different agent, action, direction, or scope)\n'
        '- Sounds like a genuine direct translation of a proverb\n'
        '- Is grammatically natural English\n\n'
        'Return ONLY: ["alternative 1", "alternative 2", "alternative 3"]'
    )

def parse_response(raw):
    if not raw: return None
    raw = re.sub(r'<reasoning>.*?</reasoning>', '', raw, flags=re.DOTALL)
    raw = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL)
    raw = raw.strip()
    raw = re.sub(r'^```\w*\n?', '', raw, flags=re.MULTILINE)
    raw = re.sub(r'\n?```$', '', raw, flags=re.MULTILINE).strip()
    try:
        d = json.loads(raw)
        if isinstance(d, list) and len(d) >= 3:
            return [str(x).strip() for x in d[:3] if str(x).strip()]
    except: pass
    m = re.search(r'\[[\s\S]*?\]', raw)
    if m:
        try:
            d = json.loads(m.group())
            if isinstance(d, list) and len(d) >= 3:
                return [str(x).strip() for x in d[:3] if str(x).strip()]
        except: pass
    matches = re.findall(r'"([^"]{8,400})"', raw)
    if len(matches) >= 3: return matches[:3]
    return None

def assemble_mcq(correct, distractors):
    rng = random.Random(42)
    choices = list(distractors[:3]) + [correct]
    rng.shuffle(choices)
    labels = ['A', 'B', 'C', 'D']
    answer = labels[choices.index(correct)]
    choice_dict = {f'Choice_{l}': choices[i] for i, l in enumerate(labels)}
    return choice_dict, answer


In [ ]:
# Cell 3: Smart API Key Rotation Callers
_key_indices = {k: 0 for k in KEYS}

def get_next_key(provider):
    if not KEYS.get(provider): return None
    k = KEYS[provider][_key_indices[provider] % len(KEYS[provider])]
    _key_indices[provider] += 1
    return k

def call_groq(sys_prompt, user_prompt, model='llama-3.3-70b-versatile', temp=0.85, max_tokens=800):
    for attempt in range(4):
        key = get_next_key('GROQ')
        if not key: return None
        try:
            r = requests.post('https://api.groq.com/openai/v1/chat/completions', headers={
                'Authorization': f'Bearer {key}'
            }, json={
                'model': model,
                'messages': [{'role':'system','content':sys_prompt}, {'role':'user','content':user_prompt}],
                'temperature': temp, 'max_tokens': max_tokens
            }, timeout=30)
            if r.status_code == 200:
                return r.json()['choices'][0]['message']['content']
            elif r.status_code == 429:
                time.sleep(2) # try next key
            else:
                pass # print(f"Groq {r.status_code}")
        except Exception as e:
            time.sleep(1)
    return None

def call_cerebras(sys_prompt, user_prompt, model='llama3.1-8b', temp=0.85, max_tokens=800):
    for attempt in range(4):
        key = get_next_key('CEREBRAS')
        if not key: return None
        try:
            r = requests.post('https://api.cerebras.ai/v1/chat/completions', headers={
                'Authorization': f'Bearer {key}'
            }, json={
                'model': model,
                'messages': [{'role':'system','content':sys_prompt}, {'role':'user','content':user_prompt}],
                'temperature': temp, 'max_tokens': max_tokens
            }, timeout=30)
            if r.status_code == 200:
                return r.json()['choices'][0]['message']['content']
            elif r.status_code == 429:
                time.sleep(4)
        except Exception as e:
            time.sleep(1)
    return None

def call_gemini(sys_prompt, user_prompt, model='gemini-2.0-flash', temp=0.85, max_tokens=800):
    for attempt in range(4):
        key = get_next_key('GEMINI')
        if not key: return None
        try:
            r = requests.post('https://generativelanguage.googleapis.com/v1beta/openai/chat/completions', headers={
                'Authorization': f'Bearer {key}'
            }, json={
                'model': model,
                'messages': [{'role':'system','content':sys_prompt}, {'role':'user','content':user_prompt}],
                'temperature': temp, 'max_tokens': max_tokens
            }, timeout=30)
            if r.status_code == 200:
                return r.json()['choices'][0]['message']['content']
            elif r.status_code == 429:
                time.sleep(2)
        except Exception as e:
            time.sleep(1)
    return None

def generate_distractors(sys_prompt, user_prompt):
    # Cascade: Try Groq 70B -> Gemini 2.0 Flash -> Cerebras
    res = call_groq(sys_prompt, user_prompt, model='llama-3.3-70b-versatile')
    if res: return res
    res = call_gemini(sys_prompt, user_prompt, model='gemini-2.0-flash')
    if res: return res
    res = call_cerebras(sys_prompt, user_prompt, model='llama3.1-8b')
    return res

def run_audit(sys_prompt, user_prompt):
    # Audit uses fast 8B models: Try Groq 8B -> Cerebras 8B
    res = call_groq(sys_prompt, user_prompt, model='llama-3.1-8b-instant', temp=0.0, max_tokens=5)
    if res: return res
    res = call_cerebras(sys_prompt, user_prompt, model='llama3.1-8b', temp=0.0, max_tokens=5)
    return res


In [ ]:
# Cell 4: Load Pilot Data
# Change to 'a' if you want to pilot Task A (literal meaning)
PILOT_TASK = 'b'
# Change to 1000 when you are ready to scale up
PILOT_N = 50

# Update this path to where your Kaggle dataset is mounted! (e.g. /kaggle/input/mcq-results)
DATA_DIR = Path('/kaggle/input/mcq-results') 
LANGUAGES = ['Yoruba','Arabic','English','French','Spanish','German']

dfs = {}
for lang in LANGUAGES:
    csv_path = DATA_DIR / f'mcq_{PILOT_TASK}_zs_{lang.lower()}.csv'
    # Fallback to local path if testing locally
    if not csv_path.exists():
        csv_path = Path(f'mcq_{PILOT_TASK}_zs_{lang.lower()}.csv')
        
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        required_cols = ['source_proverb', 'proverb_en', 'correct_meaning']
        if all(c in df.columns for c in required_cols):
            df = df.dropna(subset=required_cols).reset_index(drop=True)
            dfs[lang] = df.sample(min(PILOT_N, len(df)), random_state=42)
            print(f"✅ {lang}: Loaded {len(dfs[lang])} items")
        else:
            print(f"❌ {lang}: Missing required columns")
    else:
        print(f"❌ {lang}: File not found ({csv_path})")


In [ ]:
# Cell 5: Generation
gen_rows = []
sys_p = SYS_B_V2 if PILOT_TASK == 'b' else SYS_A_V2

for lang, df in dfs.items():
    print(f"\n[{lang}] Generating {len(df)} items...")
    fail_count = 0
    for idx, row in df.iterrows():
        proverb = str(row['source_proverb'])
        translation = str(row['proverb_en'])
        correct = str(row['correct_meaning'])
        
        if PILOT_TASK == 'b':
            user_p = make_prompt_b_v2(proverb, translation, correct, lang)
        else:
            user_p = make_prompt_a_v2(proverb, translation, lang)
            
        raw = generate_distractors(sys_p, user_p)
        dists = parse_response(raw)
        
        if not dists:
            print(f"  [PARSE FAIL] {row.get('sample_id', '?')}")
            fail_count += 1
            continue
            
        choices, answer = assemble_mcq(correct, dists)
        gen_rows.append({
            'language': lang,
            'sample_id': str(row.get('sample_id', f'{lang[:3]}{idx:04d}')),
            'source_proverb': proverb,
            'proverb_en': translation,
            'correct_meaning': correct,
            **choices,
            'Answer': answer,
            'distractor_1': dists[0],
            'distractor_2': dists[1],
            'distractor_3': dists[2],
        })
        
        if len(gen_rows) % 10 == 0:
            print(f"  ... {len(gen_rows)} items generated total")
            
    print(f"  Done {lang}. Fails: {fail_count}")
            
gen_df = pd.DataFrame(gen_rows)
gen_path = OUT_DIR / f'pilot_v2_gen_{PILOT_TASK}.csv'
gen_df.to_csv(gen_path, index=False)
print(f"\n✅ Saved {len(gen_df)} items to {gen_path}")


In [ ]:
# Cell 6: Shortcut Resistance Score (SRS) Audit
print('=== BLIND SHORTCUT AUDIT ===')
AUDIT_SYS = (
    'You are answering a multiple-choice question about a proverb. '
    'Select the option that best matches the question. '
    'Reply with ONLY the letter: A, B, C, or D.'
)
AUDIT_Q = {
    'a': 'What does this proverb most likely mean literally?',
    'b': 'Which option best captures the cultural meaning of this proverb?',
}

def audit_prompt(task, proverb, translation, ca, cb, cc, cd):
    return (
        f'Proverb: {proverb}\nEnglish: {translation}\n\n'
        f'{AUDIT_Q[task]}\nA. {ca}\nB. {cb}\nC. {cc}\nD. {cd}\n\nAnswer (A/B/C/D only):'
    )

audit_rows = []
for i, row in gen_df.iterrows():
    ap = audit_prompt(
        PILOT_TASK, row['source_proverb'], row['proverb_en'],
        row['Choice_A'], row['Choice_B'], row['Choice_C'], row['Choice_D']
    )
    raw = run_audit(AUDIT_SYS, ap)
    raw = (raw or '').strip().upper()
    pred = None
    for ch in 'ABCD':
        if raw.startswith(ch): pred = ch; break
    if not pred:
        m = re.search(r'\b([ABCD])\b', raw)
        pred = m.group(1) if m else None
        
    audit_rows.append({
        'language': row['language'],
        'sample_id': row['sample_id'],
        'correct': row['Answer'],
        'predicted': pred,
        'is_correct': int(pred == row['Answer']) if pred else 0,
    })
    
    if (i+1) % 20 == 0:
        print(f"  Audited {i+1}/{len(gen_df)}...")

audit_df = pd.DataFrame(audit_rows)
audit_path = OUT_DIR / f'pilot_v2_audit_{PILOT_TASK}.csv'
audit_df.to_csv(audit_path, index=False)

print('\n=== SHORTCUT RESISTANCE SCORE (SRS) ===')
print(f'{"Language":<12} {"N":>5} {"Acc% (SRS)":>12} {"vs 25%":>9} {"Status":>12}')
print('-'*55)
all_pass = True
for lang in LANGUAGES:
    sub = audit_df[audit_df['language']==lang]
    if len(sub)==0: continue
    acc = round(sub['is_correct'].mean()*100, 1)
    delta = acc - 25.0
    status = '✅ PASS' if acc < 40 else ('🟡 REVIEW' if acc < 55 else '🔴 FAIL')
    if acc >= 40: all_pass = False
    print(f'{lang:<12} {len(sub):>5} {acc:>10.1f}% {delta:>+8.1f}pp {status:>12}')

overall = round(audit_df['is_correct'].mean()*100, 1)
print('-'*55)
print(f'{"OVERALL":<12} {len(audit_df):>5} {overall:>10.1f}%')
print(f'\nTarget SRS: < 40% (Closer to 25% is better)')
print(f'Verdict: {"✅ READY TO SCALE" if all_pass else "🔴 NEEDS MORE WORK"}')
